In [22]:
import os
import glob
from shapely.geometry import Polygon
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.draw import polygon
from skimage import measure
import tifffile
from tqdm import tqdm
from scipy import ndimage as ndi

In [23]:
# Parquet config
file_path = r"C:\Users\jvill\Downloads\ref_cell_boundaries.parquet"
metadata_path = r"D:\ADATA Backup\HuBMAP\HuBMAP Xenium\Xenium_data\Xen24_KPMP_038_ref-selected\_metadata.csv"


metadata_list = [
'low,quality',
"POD",
"PT",
"CNT",
"DCT",
"TAL",
"DTL",
"ATL",
"IC",
"altPOD",
"altPT",
"altCNT",
"altDCT",
"altTAL",
"altDTL",
"altATL",
"altIC",
"PC",
"PEC",
"MD",
"EC",
"EC,GC",
"FIB",
"infFIB",
"pvFIB",
"aFIB",
"B",
"DC",
"MON",
"N",
"NK",
"PL",
"T",
"MAC",
"resMAC",
"moMAC",
"moMAC,INF",
"MAST",
"Lymphoid",
"Myeloid",
"NEU",
"SC,NEU",
"MC",
"MYOF",
"REN",
"VSM,P",
"unassigned"
]

In [ ]:
###  Create labeled mask 
df_metadata = pd.read_csv(metadata_path)

df = pd.read_parquet(file_path)
df_x = df['vertex_x']
df_y = df['vertex_y']

# Find resolution
diff_x = np.diff(df_x)
diff_y = np.diff(df_y)
min_x = np.min(diff_x[diff_x>0])
min_y = np.min(diff_y[diff_y>0])

# Add padding to mapping
pad_x = 10
pad_y = 10

# Convert from vertex coordinates to integer
diff_x_rounded = np.round(diff_x / min_x)
first_x = np.round(df['vertex_x'].iloc[0])
restored_x = np.concatenate(([first_x], first_x + np.cumsum(diff_x_rounded)))
restored_x = restored_x - restored_x.min() + pad_x
diff_y_rounded = np.round(diff_y / min_y)
first_y = np.round(df['vertex_y'].iloc[0])
restored_y = np.concatenate(([first_y], first_y + np.cumsum(diff_y_rounded)))
restored_y = restored_y - restored_y.min() + pad_y

max_x = int(restored_x.max()) + 1
max_y = int(restored_y.max()) + 1

df['x'] = restored_x
df['y'] = restored_y

mask = np.zeros((max_y+(2*pad_y), max_x+(2*pad_x)), dtype=np.uint8)
# Pre-calculate metadata mapping for fast O(1) lookup
label_to_idx = {label: i for i, label in enumerate(metadata_list)}
# Map cell_ID to label index directly
cell_to_idx = df_metadata.set_index('cell_ID')['levelKID_refined'].map(label_to_idx).to_dict()
# Default index for "unassigned" if missing (assuming it's at the end or handle accordingly)
unassigned_idx = label_to_idx.get('unassigned', 0)

#     # Final formatting
for label_id, group in df.groupby('label_id'):
    # Get cell_id once
    cell_id = group['cell_id'].iloc[0]
    
    # Fast lookup
    cell_label_idx = cell_to_idx.get(cell_id, unassigned_idx)

    # Draw polygon
    r = group['y'].values
    c = group['x'].values

    rr, cc = polygon(r, c, shape=mask.shape)
    mask[rr, cc] = cell_label_idx + 1

res_x = 1 / float(min_x)
res_y = 1 / float(min_y)


name = os.path.basename(file_path).split("_")[0]
output_dir = os.path.dirname(metadata_path)

tif_name = os.path.join(output_dir, f'{name}_cell_segmentation.tif')
tifffile.imwrite(
tif_name,
mask,
imagej=True,
resolution=(res_x, res_y),
resolutionunit=1,
metadata={
    'axes': 'YX',       # Stack as Z slices
    'unit':  'micron',     # Physical unit string (e.g. 'um', 'micron')
    'spacing': 1,  # Z spacing reuses Y pixel size
},
)

# Save Dataframe of coordinates
df.to_csv(os.path.join(output_dir, f'{name}_cell_segmentation.csv'))

In [27]:
# Convert labeled mask s to 
labeled_mask_path = r"D:\ADATA Backup\HuBMAP\HuBMAP Xenium\Xenium_data\Xen24_KPMP_038_ref-selected\ref_cell_segmentation-roi2-transformed.ome.tiff"
sample_name = "ref-roi2-HSI"
output_dir = os.path.join(os.path.dirname(labeled_mask_path), f"{sample_name}/")
os.makedirs(output_dir, exist_ok=True)

labeled_mask = tifffile.imread(labeled_mask_path)
resolution = 0.4971849
res = 1 / resolution
min_size = 50
for i, label in tqdm(enumerate(metadata_list,start=1)):


    mask = labeled_mask == i
    if not np.any(mask):
        continue
    
    # Remove small artifacts
    labels = measure.label(mask, connectivity=2)
    props = measure.regionprops(labels)
    filtered_mask = np.zeros_like(mask, dtype=np.uint8)
    for region in props:
        if region.area >= min_size:
            filtered_mask[labels == region.label] = 255


    mask = filtered_mask.astype(np.uint8)


    output_name = f"Xenium-{label}-mask.tif"
    tifffile.imwrite(
        os.path.join(output_dir, output_name),
        mask,
        imagej=True,
        resolution=(res, res),
        resolutionunit=1,
        metadata={
            'axes': 'YX',       # Stack as Z slices
            'unit':  'micron',     # Physical unit string (e.g. 'um', 'micron')
            'spacing': 1,  # Z spacing reuses Y pixel size
            },
    )

47it [00:39,  1.20it/s]
